# Regularization and Training Deep Networks

Deep neural networks have millions of parameters — far more than the number of training examples in many tasks. With that much capacity, they can **memorize** training data instead of learning generalizable patterns. **Regularization** constrains the model to prevent overfitting. **Training techniques** — batch normalization, careful initialization, learning rate schedules — make optimization stable and efficient.

Training a deep network is as much art as science. This notebook covers the full toolkit: regularization methods, normalization, monitoring training dynamics, and the practical workflow for training networks that generalize well.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Overfitting in Deep Networks

A deep network with enough parameters can achieve near-zero training error on almost any dataset — by memorizing individual examples including their noise. But memorization does not generalize.

**Signs of overfitting:**
- Training loss keeps decreasing while validation loss increases.
- Training accuracy much higher than validation accuracy.
- Model performs well on training data but poorly on new data.

**The bias-variance tradeoff in deep learning:**
- Underfitting (high bias) — model too simple, both train and val loss high.
- Overfitting (high variance) — model too complex, train loss low but val loss high.
- Good fit — both losses low and close together.

Regularization reduces effective model complexity without necessarily reducing parameter count.

In [ ]:
# Overfitting demonstration with increasing network capacity
X, y = make_moons(n_samples=200, noise=0.25, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

configs = [(5,), (50,), (50, 50, 50)]
labels = ["Small (5)", "Medium (50)", "Large (50-50-50)"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
xx, yy = np.meshgrid(np.linspace(-2, 3, 150), np.linspace(-1.5, 2, 150))
grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])

for ax, layers, label in zip(axes, configs, labels):
    mlp = MLPClassifier(hidden_layer_sizes=layers, max_iter=2000, random_state=42)
    mlp.fit(X_tr_s, y_tr)
    tr_acc = accuracy_score(y_tr, mlp.predict(X_tr_s))
    te_acc = accuracy_score(y_te, mlp.predict(X_te_s))
    Z = mlp.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="coolwarm", edgecolors="k", s=15)
    ax.set_title(f"{label}\ntrain={tr_acc:.2f}, test={te_acc:.2f}")

plt.suptitle("More capacity → overfitting on small dataset", y=1.02)
plt.tight_layout()
plt.show()

---

## 2. L2 Regularization (Weight Decay)

Add a penalty proportional to the squared magnitude of weights to the loss:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{data}} + \frac{\lambda}{2} \sum_l \|\mathbf{W}^{[l]}\|_F^2$$

where $\lambda$ is the regularization strength and $\|\cdot\|_F$ is the Frobenius norm (sum of squared elements).

During gradient descent, this adds a shrinkage term to each weight update:

$$\mathbf{W} \leftarrow \mathbf{W} - \eta \left(\frac{\partial \mathcal{L}}{\partial \mathbf{W}} + \lambda \mathbf{W}\right)$$

Hence the name **weight decay** — weights are pulled toward zero each step. Large weights are penalized, encouraging simpler, smoother functions.

Typical values: $\lambda = 10^{-4}$ to $10^{-2}$. In AdamW, weight decay is **decoupled** from the adaptive learning rate for better regularization.

---

## 3. Dropout

**Dropout** (Srivastava et al., 2014) randomly sets a fraction of neurons to zero during each training step. Typically 20–50% of neurons are dropped in hidden layers.

During training:

$$\mathbf{a}_{\text{dropped}} = \mathbf{m} \odot \mathbf{a}$$

where $\mathbf{m}$ is a binary mask with each element independently set to 0 with probability $p$ (dropout rate) and 1 with probability $1 - p$.

During inference, all neurons are active but outputs are scaled by $(1 - p)$ to maintain expected activation magnitude (inverted dropout scales during training instead).

**Why dropout works:**
- Prevents **co-adaptation** — neurons cannot rely on specific other neurons being present.
- Acts as an **ensemble** of exponentially many thinned networks sharing weights.
- Adds noise that acts as regularization.

Dropout is applied to fully connected layers. Convolutional layers often use spatial dropout or rely on batch normalization instead.

In [ ]:
def dropout(a, p=0.5, training=True):
    """Inverted dropout: scale during training."""
    if not training:
        return a
    mask = (np.random.rand(*a.shape) > p).astype(float)
    return a * mask / (1 - p)

# Demonstrate dropout effect on activations
a = np.ones((4, 8))  # 4 neurons, batch of 8

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, p, title in zip(axes, [0.0, 0.3, 0.5], ["No dropout", "p=0.3", "p=0.5"]):
    dropped = dropout(a, p=p, training=True)
    ax.imshow(dropped, cmap="RdYlGn", vmin=0, vmax=2)
    ax.set_title(f"{title}\n{int((dropped==0).sum())}/{dropped.size} zeroed")
    ax.set_xlabel("Batch samples")
    ax.set_ylabel("Neurons")
plt.suptitle("Dropout randomly zeros neurons during training", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# L2 regularization vs no regularization (sklearn MLP uses alpha for L2)
print(f"{'alpha (L2)':>12s} {'Train acc':>10s} {'Test acc':>10s}")
print("-" * 34)
for alpha in [0.0, 0.001, 0.01]:
    mlp = MLPClassifier(hidden_layer_sizes=(100, 50), alpha=alpha,
                         max_iter=2000, random_state=42)
    mlp.fit(X_tr_s, y_tr)
    tr = accuracy_score(y_tr, mlp.predict(X_tr_s))
    te = accuracy_score(y_te, mlp.predict(X_te_s))
    print(f"{alpha:12.4f} {tr:10.3f} {te:10.3f}")

print("\nDropout is typically added as a layer in deep learning frameworks:")
print("  nn.Dropout(p=0.5)  or  tf.keras.layers.Dropout(0.5)")

---

## 4. Early Stopping

Monitor validation loss during training. Stop when it stops improving — before the model overfits.

```
best_val_loss = ∞
patience_counter = 0

for epoch in range(max_epochs):
    train_one_epoch()
    val_loss = evaluate_validation()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint()       # keep best model
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        break                   # stop training

load_best_checkpoint()          # use best model, not last
```

**Patience** — how many epochs to wait without improvement before stopping (typically 5–20). Early stopping is simple, effective, and requires no modification to the model architecture.

In [ ]:
# Simulated training curves: where to stop
epochs = np.arange(100)
train_loss = 2.0 * np.exp(-0.05 * epochs) + 0.05 + 0.02 * np.random.randn(100)
val_loss = 2.0 * np.exp(-0.04 * epochs) + 0.1 + 0.03 * epochs / 100 + 0.03 * np.random.randn(100)

best_epoch = np.argmin(val_loss)

plt.figure(figsize=(9, 4))
plt.plot(epochs, train_loss, "b-", label="Train loss", linewidth=1.5)
plt.plot(epochs, val_loss, "r-", label="Val loss", linewidth=1.5)
plt.axvline(best_epoch, color="green", linestyle="--",
            label=f"Early stop (epoch {best_epoch})")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Stop when validation loss stops improving")
plt.legend()
plt.show()

---

## 5. Batch Normalization

**Batch Normalization** (Ioffe & Szegedy, 2015) normalizes the inputs to each layer:

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

$$y_i = \gamma \hat{x}_i + \beta$$

where $\mu_B$ and $\sigma_B^2$ are the mean and variance computed over the current mini-batch, and $\gamma$, $\beta$ are **learnable** scale and shift parameters that let the network undo normalization if needed.

**Benefits:**
- Reduces **internal covariate shift** — distribution of layer inputs changes less during training.
- Allows **higher learning rates** — training is more stable.
- Acts as a **regularizer** — mini-batch statistics add noise.
- Reduces sensitivity to **weight initialization**.

During inference, use running averages of $\mu$ and $\sigma$ computed during training (not batch statistics).

Batch norm is placed **before** the activation function in most modern architectures: Linear → BatchNorm → ReLU.

In [ ]:
def batch_norm(x, gamma, beta, eps=1e-5):
    """Batch normalization on a single feature across batch."""
    mu = x.mean()
    sigma = x.std()
    x_norm = (x - mu) / (sigma + eps)
    return gamma * x_norm + beta

# Before and after batch norm
raw = np.random.randn(64) * 5 + 10  # mean=10, std=5
normalized = batch_norm(raw, gamma=1.0, beta=0.0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(raw, bins=20, color="steelblue", alpha=0.8)
axes[0].set_title(f"Before BN: μ={raw.mean():.1f}, σ={raw.std():.1f}")
axes[1].hist(normalized, bins=20, color="coral", alpha=0.8)
axes[1].set_title(f"After BN: μ={normalized.mean():.1f}, σ={normalized.std():.1f}")
for ax in axes:
    ax.set_xlabel("Activation value")
plt.suptitle("Batch normalization standardizes activations", y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Other Normalization Techniques

| Method | Normalizes Over | Use Case |
|--------|----------------|----------|
| **Batch Norm** | Mini-batch | CNNs, large batch sizes |
| **Layer Norm** | Features within one sample | Transformers, RNNs, small batches |
| **Instance Norm** | Spatial positions per channel | Style transfer |
| **Group Norm** | Groups of channels | Small batch sizes where batch norm fails |

**Layer Normalization** is the default in Transformers because sequence lengths vary and batch sizes can be small. It normalizes across features rather than across the batch.

---

## 7. Data Augmentation

For image and text data, **augmentation** artificially expands the training set by applying label-preserving transformations:

- Images: flip, rotate, crop, color jitter, cutout, mixup.
- Text: back-translation, synonym replacement, random insertion/deletion.
- Audio: time stretch, pitch shift, add noise.

Augmentation forces the model to learn invariant features — a cat is still a cat when flipped or slightly rotated. It is one of the most effective regularization techniques for vision tasks.

---

## 8. Label Smoothing

Instead of hard labels $[0, 0, 1, 0]$, use soft labels $[0.02, 0.02, 0.92, 0.02]$. This prevents the model from becoming **overconfident** — pushing logits to extreme values to match hard 0/1 targets.

$$y_{\text{smooth}} = (1 - \alpha) \cdot y_{\text{hard}} + \frac{\alpha}{K}$$

where $\alpha$ is the smoothing factor (typically 0.1) and $K$ is the number of classes.

Label smoothing improves calibration (predicted probabilities match actual frequencies) and generalization. Widely used in training image classifiers and language models.

In [ ]:
# Label smoothing example
K = 5  # classes
true_class = 2
alpha = 0.1

hard_label = np.zeros(K)
hard_label[true_class] = 1.0

smooth_label = (1 - alpha) * hard_label + alpha / K

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(K), hard_label, color="steelblue")
axes[0].set_title("Hard label")
axes[0].set_xlabel("Class")
axes[0].set_ylim(0, 1.1)
axes[1].bar(range(K), smooth_label, color="coral")
axes[1].set_title(f"Smoothed label (α={alpha})")
axes[1].set_xlabel("Class")
axes[1].set_ylim(0, 1.1)
plt.suptitle("Label smoothing prevents overconfident predictions", y=1.02)
plt.tight_layout()
plt.show()

---

## 9. Gradient Clipping

In deep networks and RNNs, gradients can explode during training. **Gradient clipping** caps the gradient magnitude:

**By value:**

$$g_i \leftarrow \text{clip}(g_i, -\text{threshold}, +\text{threshold})$$

**By norm (more common):**

If $\|\mathbf{g}\| > \text{max\_norm}$, scale: $\mathbf{g} \leftarrow \mathbf{g} \cdot \frac{\text{max\_norm}}{\|\mathbf{g}\|}$

Gradient clipping does not change the gradient direction — only its magnitude. Essential for training RNNs, Transformers, and large language models.

In [ ]:
def clip_grad_by_norm(grad, max_norm=1.0):
    norm = np.linalg.norm(grad)
    if norm > max_norm:
        return grad * (max_norm / norm)
    return grad

gradients = [np.array([3.0, 4.0]), np.array([0.1, 0.2]), np.array([10.0, -10.0])]

print(f"{'Gradient':>20s} {'Norm':>8s} {'Clipped':>20s} {'New Norm':>10s}")
print("-" * 62)
for g in gradients:
    clipped = clip_grad_by_norm(g, max_norm=1.0)
    print(f"{str(g):>20s} {np.linalg.norm(g):8.3f} {str(np.round(clipped, 3)):>20s} {np.linalg.norm(clipped):10.3f}")

---

## 10. Weight Initialization

Poor initialization can prevent training entirely. The right scale depends on the activation function and layer size.

**Xavier (Glorot) initialization** — for sigmoid/tanh:

$$W \sim \mathcal{U}\left(-\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}, \sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}\right)$$

**He initialization** — for ReLU:

$$W \sim \mathcal{N}\left(0, \frac{2}{n_{\text{in}}}\right)$$

Biases are typically initialized to zero. Proper initialization ensures activations and gradients have reasonable magnitude at the start of training.

---

## 11. Learning Rate and Schedules

The learning rate is the most important hyperparameter. Combined with schedules:

| Schedule | Description | When to Use |
|----------|-------------|-------------|
| **Constant** | Fixed $\eta$ throughout | Simple baselines |
| **Step decay** | Reduce by factor every N epochs | General purpose |
| **Cosine annealing** | Smooth cosine curve to near-zero | Modern vision training |
| **Warmup + decay** | Linear increase then decay | Transformers, large batches |
| **Reduce on plateau** | Decrease when val loss stalls | When optimal $\eta$ is unknown |

**Learning rate finder** — start with a tiny $\eta$, increase exponentially each batch, plot loss vs $\eta$. Choose $\eta$ just before loss starts increasing.

---

## 12. Monitoring Training

Track these metrics during training:

- **Training loss** — should decrease steadily.
- **Validation loss** — should decrease, then may plateau or increase (overfitting signal).
- **Training accuracy** — classification tasks.
- **Validation accuracy** — the metric that matters for model selection.
- **Learning rate** — if using a schedule.
- **Gradient norms** — sudden spikes indicate instability.
- **Weight distributions** — should not collapse to zero or explode.

Tools like TensorBoard, Weights & Biases, and MLflow log these metrics for visualization and comparison across experiments.

In [ ]:
# L2 regularization strength comparison
alphas = [0, 0.0001, 0.01, 1.0]
train_accs, test_accs = [], []

for alpha in alphas:
    mlp = MLPClassifier(hidden_layer_sizes=(100, 50), alpha=alpha,
                         max_iter=2000, random_state=42)
    mlp.fit(X_tr_s, y_tr)
    train_accs.append(accuracy_score(y_tr, mlp.predict(X_tr_s)))
    test_accs.append(accuracy_score(y_te, mlp.predict(X_te_s)))

x = np.arange(len(alphas))
width = 0.35
plt.figure(figsize=(8, 4))
plt.bar(x - width/2, train_accs, width, label="Train", color="steelblue")
plt.bar(x + width/2, test_accs, width, label="Test", color="coral")
plt.xticks(x, [str(a) for a in alphas])
plt.xlabel("L2 regularization (alpha)")
plt.ylabel("Accuracy")
plt.title("Regularization reduces overfitting gap")
plt.legend()
plt.show()

---

## 13. The Complete Regularization Toolkit

| Technique | Type | Effect |
|-----------|------|--------|
| L2 weight decay | Penalty | Shrinks weights |
| Dropout | Architecture | Random neuron removal |
| Early stopping | Training | Stop before overfitting |
| Batch normalization | Normalization | Stabilize + regularize |
| Data augmentation | Data | Expand effective dataset |
| Label smoothing | Target | Prevent overconfidence |
| Gradient clipping | Optimization | Prevent explosion |
| Weight initialization | Setup | Stable start |

In practice, use **multiple techniques together**. A typical vision training recipe: He initialization + batch norm + data augmentation + dropout + weight decay + early stopping + cosine learning rate schedule.

---

## 14. Training Workflow Checklist

1. **Prepare data** — split train/val/test, normalize inputs, augment training data.
2. **Initialize** — He/Xavier weights, zero biases.
3. **Choose optimizer** — Adam or AdamW with weight decay.
4. **Set learning rate** — use learning rate finder or start with 1e-3.
5. **Add regularization** — dropout, batch norm, L2 as appropriate.
6. **Train with monitoring** — log train/val loss and accuracy each epoch.
7. **Early stopping** — save best checkpoint based on validation loss.
8. **Evaluate** — test set, once, with best checkpoint.
9. **Iterate** — adjust architecture, regularization, or learning rate based on diagnostics.

---

## 15. Summary

Deep networks need regularization to generalize. **L2 weight decay** shrinks weights. **Dropout** prevents co-adaptation. **Early stopping** halts training at the right moment. **Batch normalization** stabilizes training and acts as a regularizer. **Data augmentation** and **label smoothing** further improve generalization.

**Gradient clipping** prevents exploding gradients. **Careful initialization** ensures training starts in a healthy regime. **Learning rate schedules** fine-tune convergence.

Training deep networks is iterative — monitor diagnostics, apply multiple regularization techniques, and select the checkpoint that performs best on validation data. The goal is not zero training error; it is the best possible performance on unseen data.